# 08 — Observability: Health, Metrics, State, and Visualization

Multigen exposes three observability surfaces:

| Surface | API | Source | What it shows |
|---------|-----|--------|---------------|
| **Health** | `get_health(id)` | Live Temporal workflow | interrupt flag, CB status, errors, dead letters |
| **Metrics** | `get_metrics(id)` | Live Temporal workflow | execution counters, reflection/fan-out counts |
| **State** | `get_state(id)` | MongoDB (CQRS read model) | All node outputs as persisted documents |

## Notebook Goal

1. Run a complete graph workflow (4-node resume pipeline)
2. Poll `get_health()` every 2 seconds during execution — observe live CB state
3. After completion: call `get_metrics()` for execution counters
4. Call `get_state()` to load all node outputs from MongoDB
5. Build a **pandas DataFrame** from node results
6. Render a **matplotlib bar chart** showing nodes_executed, reflections_triggered, fan_outs_executed

In [ ]:
import time
import threading
import pprint

from multigen import SyncMultigenClient
from multigen.dsl import GraphBuilder

ORCHESTRATOR_URL = "http://localhost:8009"
client = SyncMultigenClient(base_url=ORCHESTRATOR_URL, timeout=60.0)
assert client.ping(), "Orchestrator not reachable."
print("Connected to Multigen orchestrator.")

## 1. Build a Rich Graph Workflow

We use a 4-node pipeline with:
- Reflection on `match_skills` (threshold=0.85) to potentially trigger reflection metrics
- A `GuardrailAgent` node with a fallback to observe CB fields in health

In [ ]:
graph_def = (
    GraphBuilder()
    .node("parse_resume")
        .agent("ResumeParserAgent")
        .params(output_format="json", extract_skills=True)
        .timeout(30)
        .done()
    .node("match_skills")
        .agent("SkillMatcherAgent")
        .params(job_title="DevOps Engineer", min_match_score=0.5)
        .reflect(threshold=0.85, max_rounds=2, critic="EchoAgent")
        .timeout(45)
        .done()
    .node("guardrail")
        .agent("GuardrailAgent")
        .params(policy="general_compliance")
        .fallback("EchoAgent")
        .timeout(20)
        .done()
    .node("compile_report")
        .agent("ReportCompilerAgent")
        .params(format="json", include_recommendations=True)
        .timeout(30)
        .done()
    .edge("parse_resume",  "match_skills")
    .edge("match_skills",  "guardrail")
    .edge("guardrail",     "compile_report")
    .entry("parse_resume")
    .max_cycles(15)
    .circuit_breaker(trip_threshold=3, recovery_executions=5)
    .build()
)

print("Graph built — nodes:", [n["id"] for n in graph_def["nodes"]])
print("CB config:", graph_def["circuit_breaker"])

## 2. Launch the Workflow

In [ ]:
payload = {
    "resume_text": (
        "Frank Mueller, 8 years DevOps/SRE, Kubernetes, Terraform, "
        "Prometheus, Grafana, ArgoCD, AWS, GCP. CKA certified."
    ),
    "candidate_id": "candidate-007",
}

response = client.run_graph(graph_def=graph_def, payload=payload)
instance_id = response.instance_id
print(f"Launched — instance_id: {instance_id}")

## 3. Live Health Polling During Execution

We poll `get_health()` every 2 seconds while the graph executes. This lets us observe:
- `interrupted` flag (paused by operator or error)
- `cb_trips_total` (circuit breaker activations)
- `pending_count` (nodes in the execution queue)
- `errors` (any runtime errors)

The polling loop stops once `pending_count` drops to 0 (all nodes complete) or after a timeout.

In [ ]:
health_snapshots = []
start_time = time.time()

print(f"{'Elapsed':>8s}  {'interrupted':>12s}  {'pending':>8s}  {'cb_trips':>9s}  {'errors'}")
print("-" * 65)

for _ in range(20):  # poll for up to 40 seconds
    try:
        h = client.get_health(instance_id)
        elapsed = time.time() - start_time
        health_snapshots.append({
            "elapsed": round(elapsed, 1),
            "interrupted": h.interrupted,
            "pending_count": h.pending_count,
            "cb_trips_total": h.cb_trips_total,
            "error_count": len(h.errors),
        })
        print(
            f"{elapsed:8.1f}s  "
            f"{str(h.interrupted):>12s}  "
            f"{h.pending_count:>8d}  "
            f"{h.cb_trips_total:>9d}  "
            f"{h.errors or 'none'}"
        )
        # Stop when all nodes are done
        if h.pending_count == 0 and elapsed > 5:
            print("\nAll pending nodes completed — stopping health poll.")
            break
    except Exception as e:
        print(f"  (health unavailable: {e})")
    time.sleep(2)

## 4. Final Metrics After Completion

In [ ]:
metrics = client.get_metrics(instance_id)

print("Final Execution Metrics")
print("=" * 40)
metric_data = {
    "nodes_executed":        metrics.nodes_executed,
    "nodes_skipped":         metrics.nodes_skipped,
    "reflections_triggered": metrics.reflections_triggered,
    "fan_outs_executed":     metrics.fan_outs_executed,
    "circuit_breaker_trips": metrics.circuit_breaker_trips,
    "error_count":           metrics.error_count,
    "dead_letter_count":     metrics.dead_letter_count,
}
for key, val in metric_data.items():
    print(f"  {key:<28s}: {val}")

## 5. Load All Node Outputs from MongoDB State

`get_state()` reads all persisted node outputs from the MongoDB CQRS read model. This gives a complete audit trail of what each node returned.

In [ ]:
workflow_state = client.get_state(instance_id)

print(f"MongoDB State Snapshot")
print(f"  workflow_id  : {workflow_state.workflow_id}")
print(f"  node_count   : {workflow_state.count}")
print()

for ns in workflow_state.nodes:
    print(f"  [{ns.node_id}]  updated_at={ns.updated_at}")
    output_keys = list(ns.output.keys()) if ns.output else []
    print(f"    output keys: {output_keys}")

## 6. Build a pandas DataFrame of Node Results

We flatten each node's output into a row in a DataFrame for easy tabular inspection.

In [ ]:
try:
    import pandas as pd

    rows = []
    for ns in workflow_state.nodes:
        row = {
            "node_id": ns.node_id,
            "updated_at": ns.updated_at,
            "output_keys": ", ".join(ns.output.keys()) if ns.output else "",
            "confidence": ns.output.get("confidence", None),
            "score": ns.output.get("score", ns.output.get("total_score", None)),
            "error": ns.output.get("error", None),
        }
        rows.append(row)

    df = pd.DataFrame(rows)
    print("Node Results DataFrame:")
    print(df.to_string(index=False))
except ImportError:
    print("pandas not installed — skipping DataFrame. Install with: pip install pandas")
    df = None

## 7. Visualize Health Poll Timeline

Plot how `pending_count` and `cb_trips_total` changed over the execution timeline.

In [ ]:
try:
    import matplotlib.pyplot as plt
    import matplotlib.gridspec as gridspec

    if not health_snapshots:
        print("No health snapshots collected — skipping chart.")
    else:
        elapsed_times = [s["elapsed"] for s in health_snapshots]
        pending_counts = [s["pending_count"] for s in health_snapshots]
        cb_trips = [s["cb_trips_total"] for s in health_snapshots]

        fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(10, 6), sharex=True)

        # Pending count over time
        ax1.step(elapsed_times, pending_counts, where="post", color="#1976D2", linewidth=2)
        ax1.fill_between(elapsed_times, pending_counts, step="post", alpha=0.2, color="#1976D2")
        ax1.set_ylabel("Pending Node Count", fontsize=11)
        ax1.set_title(f"Live Health Poll Timeline\n(workflow: {instance_id})", fontsize=12)
        ax1.grid(alpha=0.3)
        ax1.set_ylim(bottom=0)

        # Circuit breaker trips over time
        ax2.step(elapsed_times, cb_trips, where="post", color="#d32f2f", linewidth=2)
        ax2.fill_between(elapsed_times, cb_trips, step="post", alpha=0.2, color="#d32f2f")
        ax2.set_ylabel("CB Trips (cumulative)", fontsize=11)
        ax2.set_xlabel("Elapsed Time (seconds)", fontsize=11)
        ax2.grid(alpha=0.3)
        ax2.set_ylim(bottom=0)

        plt.tight_layout()
        plt.savefig("health_poll_timeline.png", dpi=120, bbox_inches="tight")
        plt.show()
        print("Saved to health_poll_timeline.png")
except ImportError:
    print("matplotlib not installed — skipping chart. Install with: pip install matplotlib")

## 8. Metrics Bar Chart

A bar chart comparing the main execution metrics: nodes executed, reflections triggered, fan-outs, CB trips, and errors.

In [ ]:
try:
    import matplotlib.pyplot as plt
    import numpy as np

    # Chart data — key execution counters
    chart_metrics = [
        ("nodes_executed",        metric_data["nodes_executed"],        "#388e3c"),
        ("nodes_skipped",         metric_data["nodes_skipped"],         "#fbc02d"),
        ("reflections_triggered", metric_data["reflections_triggered"], "#1976D2"),
        ("fan_outs_executed",     metric_data["fan_outs_executed"],     "#7b1fa2"),
        ("circuit_breaker_trips", metric_data["circuit_breaker_trips"], "#d32f2f"),
        ("error_count",           metric_data["error_count"],           "#e64a19"),
    ]

    labels = [m[0] for m in chart_metrics]
    values = [m[1] for m in chart_metrics]
    colors = [m[2] for m in chart_metrics]

    fig, ax = plt.subplots(figsize=(11, 5))
    bars = ax.bar(labels, values, color=colors, edgecolor="black", linewidth=0.7)

    # Annotate each bar with its value
    for bar, val in zip(bars, values):
        ax.text(
            bar.get_x() + bar.get_width() / 2,
            bar.get_height() + 0.05,
            str(val),
            ha="center", va="bottom", fontweight="bold", fontsize=12,
        )

    ax.set_ylabel("Count", fontsize=12)
    ax.set_title(
        f"Multigen Execution Metrics\n(workflow: {instance_id[:20]}...)",
        fontsize=13,
    )
    ax.set_ylim(0, max(values) + 2 if max(values) > 0 else 5)
    ax.set_xticks(range(len(labels)))
    ax.set_xticklabels(labels, rotation=25, ha="right", fontsize=10)
    ax.grid(axis="y", alpha=0.3)

    plt.tight_layout()
    plt.savefig("execution_metrics.png", dpi=120, bbox_inches="tight")
    plt.show()
    print("Saved to execution_metrics.png")
except ImportError:
    print("matplotlib not installed — skipping chart. Install with: pip install matplotlib")

## 9. Health Object Field Reference

```python
WorkflowHealth(
    workflow_id    = "abc-123",         # instance ID
    interrupted    = False,             # True if interrupt signal was sent
    pending_count  = 0,                 # nodes queued but not yet executed
    skip_nodes     = [],                # node IDs marked for skip
    cb_trips_total = 0,                 # cumulative circuit-breaker activations
    errors         = [],                # recent error messages
    dead_letters   = [],                # node IDs that exhausted retries+fallback
)
```

## WorkflowMetrics Field Reference

```python
WorkflowMetrics(
    workflow_id            = "abc-123",
    nodes_executed         = 4,   # total agent invocations
    nodes_skipped          = 0,   # skipped via skip_node signal
    reflections_triggered  = 1,   # reflection loops that fired
    fan_outs_executed      = 0,   # fan-out groups completed
    circuit_breaker_trips  = 0,   # times CB switched to OPEN
    error_count            = 0,   # total errors across all nodes
    dead_letter_count      = 0,   # nodes sent to dead-letter queue
)
```

## Observability Best Practices

1. **Always poll `get_health()`** during long-running workflows — catches CB trips and errors early
2. **Use `get_state()` for audit** — MongoDB state is the persistent record of what happened
3. **Export metrics to your APM** — add Prometheus scraping or call `get_metrics()` on a schedule
4. **Alert on `dead_letter_count > 0`** — dead letters mean nodes failed all retries and fallbacks
5. **Alert on `cb_trips_total` rising** — indicates an underlying agent is consistently failing

In [ ]:
client.close()
print("All done! Observability notebook complete.")

---

## Notebook Series Complete

You have now covered all major features of the Multigen framework:

| Notebook | Topic |
|----------|-------|
| 01 | Quickstart: connect, ping, run EchoAgent |
| 02 | Graph workflows: 4-node resume pipeline |
| 03 | Runtime signals: interrupt, inject, jump, skip, resume |
| 04 | Circuit breakers: automatic failure isolation with fallbacks |
| 05 | Reflection loops: confidence-based self-improvement |
| 06 | Fan-out and consensus: parallel agent reasoning |
| 07 | Dynamic agents: blueprint-based on-the-fly agent creation |
| **08** | **Observability: health polling, metrics, state, charts** |

For more, see the [Multigen documentation](../docs/) or the [examples directory](../examples/).